In [ ]:
# ライブラリ読み込み
import os
import numpy as np
import pandas as pd
import scipy.io
from scipy.io import mmread, mmwrite
from scipy.sparse import csr_matrix
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ann
import copy 

In [ ]:
datadir = 'mtscATAC-seq/rds_output'
expname = ['bulk1', 'bulk2', 'PT1_neg', 'PT1_pos', 'PT2_neg', 'PT2_pos']
matname = ['A_counts_fw', 'A_counts_rev', 'C_counts_fw', 'C_counts_rev', 'G_counts_fw', 'G_counts_rev', 'T_counts_fw', 'T_counts_rev', 'coverage']
adatas = []
adatas_mg = []


for x in range(len(expname)):
    genes = pd.read_csv(os.path.join(datadir,expname[x],'genes.tsv'), sep="\t")
    barcodes = np.loadtxt(os.path.join(datadir,expname[x],'barcodes.tsv'), dtype='str')
    mtx = scipy.io.mmread(os.path.join(datadir,expname[x],'matrix.mtx'))
    mtx = mtx.toarray()
    mtx = mtx.T
    adata = ann.AnnData(X=mtx, obs=barcodes, var=genes)
    adata.var_names_make_unique()
    adatas.append(adata)

In [ ]:
q = pd.read_table()

In [ ]:
adatas_mg = []

for x in range(len(expname)):
    genes = pd.read_csv(os.path.join(datadir, expname[x] + '_mg', 'genes.tsv'), sep="\t")
    barcodes = np.loadtxt(os.path.join(datadir, expname[x] + '_mg', 'barcodes.tsv'), dtype='str')
    mtx = scipy.io.mmread(os.path.join(datadir, expname[x] + '_mg', 'coverage.mtx'))
    mtx = mtx.toarray()
    mtx = mtx.T
    adata_mg = ann.AnnData(X=mtx, obs=barcodes, var=genes)
    for y in range(len(matname)):
        mtx = scipy.io.mmread(os.path.join(datadir, expname[x] + '_mg', matname[y]+'.mtx'))
        mtx = mtx.toarray()
        mtx = mtx.T
        adata_mg.layers[matname[y]] = mtx
    adata_mg.var_names_make_unique() # this is unnecessary if using `var_names='gene_ids'` in `sc.read_10x_mtx`
    adatas_mg.append(adata_mg)

In [ ]:
for x in range(len(expname)):
    adatas[x].obs = adatas[x].obs.rename(columns={0:'barcode'})
    adatas[x].var = adatas[x].var.drop(columns='strand')
    adatas_mg[x].obs = adatas_mg[x].obs.rename(columns={0:'barcode'})
    adatas_mg[x].var = adatas_mg[x].var.drop(columns=['strand','width'])

In [ ]:
# ref Alleleの行番号以外の行番号を抽出

not_A_index = np.array(adatas_mg[0].var.reset_index().query('refAllele!="A"').index)
not_C_index = np.array(adatas_mg[0].var.reset_index().query('refAllele!="C"').index)
not_G_index = np.array(adatas_mg[0].var.reset_index().query('refAllele!="G"').index)
not_T_index = np.array(adatas_mg[0].var.reset_index().query('refAllele!="T"').index)

In [ ]:
adatas_mg

In [ ]:
adatas

In [ ]:
adatas_mg_VAF = copy.deepcopy(adatas_mg)

for x in range(len(expname)):
    del adatas_mg_VAF[x].layers
    
    #adatas_mg_VAF[x].layers['variant_refA'] = adatas_mg[x].X - adatas_mg[x].layers['C_counts_fw'] - adatas_mg[x].layers['C_counts_rev'] - adatas_mg[x].layers['G_counts_fw'] - adatas_mg[x].layers['G_counts_rev'] - adatas_mg[x].layers['T_counts_fw'] - adatas_mg[x].layers['T_counts_rev']
    adatas_mg_VAF[x].layers['variant_refA'] = adatas_mg[x].X - adatas_mg[x].layers['A_counts_fw'] - adatas_mg[x].layers['A_counts_rev']
    adatas_mg_VAF[x].layers['variant_refA'][:,not_A_index] = 0
    
    #adatas_mg_VAF[x].layers['variant_refC'] = adatas_mg[x].X - adatas_mg[x].layers['A_counts_fw'] - adatas_mg[x].layers['A_counts_rev'] - adatas_mg[x].layers['G_counts_fw'] - adatas_mg[x].layers['G_counts_rev'] - adatas_mg[x].layers['T_counts_fw'] - adatas_mg[x].layers['T_counts_rev']
    adatas_mg_VAF[x].layers['variant_refC'] = adatas_mg[x].X - adatas_mg[x].layers['C_counts_fw'] - adatas_mg[x].layers['C_counts_rev']
    adatas_mg_VAF[x].layers['variant_refC'][:,not_C_index] = 0
    
    #adatas_mg_VAF[x].layers['variant_refG'] = adatas_mg[x].X - adatas_mg[x].layers['A_counts_fw'] - adatas_mg[x].layers['A_counts_rev'] - adatas_mg[x].layers['C_counts_fw'] - adatas_mg[x].layers['C_counts_rev'] - adatas_mg[x].layers['G_counts_fw'] - adatas_mg[x].layers['G_counts_rev']
    adatas_mg_VAF[x].layers['variant_refG'] = adatas_mg[x].X - adatas_mg[x].layers['G_counts_fw'] - adatas_mg[x].layers['G_counts_rev']
    adatas_mg_VAF[x].layers['variant_refG'][:,not_G_index] = 0
    
    #adatas_mg_VAF[x].layers['variant_refT'] = adatas_mg[x].X - adatas_mg[x].layers['A_counts_fw'] - adatas_mg[x].layers['A_counts_rev'] - adatas_mg[x].layers['C_counts_fw'] - adatas_mg[x].layers['C_counts_rev'] - adatas_mg[x].layers['G_counts_fw'] - adatas_mg[x].layers['G_counts_rev']
    adatas_mg_VAF[x].layers['variant_refT'] = adatas_mg[x].X - adatas_mg[x].layers['T_counts_fw'] - adatas_mg[x].layers['T_counts_rev']
    adatas_mg_VAF[x].layers['variant_refT'][:,not_T_index] = 0
    
    adatas_mg_VAF[x].layers['all_mut'] = adatas_mg_VAF[x].layers['variant_refA'] + adatas_mg_VAF[x].layers['variant_refC'] + adatas_mg_VAF[x].layers['variant_refG'] + adatas_mg_VAF[x].layers['variant_refT']
    adatas_mg_VAF[x].layers['VAF'] = adatas_mg_VAF[x].layers['all_mut'] / adatas_mg[x].X

In [ ]:
layername = ['variant_refA', 'variant_refC', 'variant_refG', 'variant_refT', 'all_mut', 'VAF']

for x in range(len(expname)):
    for y in range(len(layername)-2):
        del adatas_mg_VAF[x].layers[layername[y]]

In [ ]:
adatas[0].X

In [ ]:
adatas_mg_VAF[3].X

In [ ]:
for x in range(len(expname)):
    adatas_mg_VAF[x].layers['VAF'] = np.nan_to_num(adatas_mg_VAF[x].layers['VAF'])

In [ ]:
len(adatas_mg_VAF[3].var)

In [ ]:
#データの中に0を含む割合を表示
for x in range(len(expname)):
    a = len(adatas[x].obs)*len(adatas[x].var)
    print(np.count_nonzero(adatas[x].X == 0)/a*100)

In [ ]:
#データの中に0を含む割合を表示
for x in range(len(expname)):
    a = len(adatas_mg_VAF[x].obs)*len(adatas_mg_VAF[x].var)
    print(np.count_nonzero(adatas_mg_VAF[x].X == 0)/a*100)

In [ ]:
#データの中に0を含む割合を表示
for x in range(len(expname)):
    a = len(adatas_mg_VAF[x].obs)*len(adatas_mg_VAF[x].var)
    print(np.count_nonzero(adatas_mg_VAF[x].layers['all_mut'] == 0)/a*100)

In [ ]:
6000 * 16569

In [ ]:
# celltype annotation


In [ ]:
layername = ['all_mut', 'VAF']

for x in range(len(expname)):
    adatas[x].X = csr_matrix(adatas[x].X)
    for y in range(len(layername)):
        adatas_mg_VAF[x].layers[layername[y]] = csr_matrix(adatas_mg_VAF[x].layers[layername[y]])

In [ ]:
dirname_h5ad = 'mtscATAC-seq/h5ad_files'
if not os.path.exists(dirname_h5ad):
    os.makedirs(dirname_h5ad)
    
for x in range(len(expname)):
    adatas[x].write(os.path.join(dirname_h5ad, expname[x] + '.h5ad'), compression='gzip')
    adatas_mg_VAF[x].write(os.path.join(dirname_h5ad, expname[x] + '_mg.h5ad'), compression='gzip')

In [ ]:
#dirname_coo = 'mtscATAC-seq/coo_version'
#if not os.path.exists(dirname_coo):
#    os.makedirs(dirname_coo)
#
#for x in range(len(expname)):
#    if not os.path.exists(os.path.join(dirname_coo, expname[x])):
#        os.makedirs(os.path.join(dirname_coo, expname[x]))
#    if not os.path.exists(os.path.join(dirname_coo, expname[x], expname[x])):
#        os.makedirs(os.path.join(dirname_coo, expname[x]))
#    if not os.path.exists(os.path.join(dirname_coo, expname[x]+'_mg')):
#        os.makedirs(os.path.join(dirname_coo, expname[x]+'_mg'))
#    
#    adatas[x].var.to_csv(os.path.join(dirname_coo, expname[x], expname[x], 'genes.tsv'),sep='\t')
#    adatas[x].obs.to_csv(os.path.join(dirname_coo, expname[x], expname[x], 'barcode.tsv'),sep='\t')
#    mmwrite(os.path.join(dirname_coo, expname[x], expname[x], 'matrix'), adatas[x].X)
#    
#    adatas_mg[x].var.to_csv(os.path.join(dirname_coo, expname[x], expname[x]+'_mg', 'genes.tsv'),sep='\t')
#    adatas_mg[x].obs.to_csv(os.path.join(dirname_coo, expname[x], expname[x]+'_mg', 'barcode.tsv'),sep='\t')
#    mmwrite(os.path.join(dirname_coo, expname[x], expname[x]+'_mg', 'coverage'),adatas_mg[x].X)
#    for y in range(0,len(matname)-1,1):
#        mmwrite(os.path.join(dirname_coo, expname[x], expname[x]+'_mg', matname[y]),adatas_mg[x].layers[matname[y]])

In [ ]:
#for x in range(len(expname)):
#    scipy.sparse.save_npz(os.path.join(dirname_coo, expname[x], expname[x], 'matrix.npz'), adatas[x].X)
#    scipy.sparse.save_npz(os.path.join(dirname_coo, expname[x], expname[x]+'_mg', 'coverage.npz'), adatas_mg[x].X)
#    
#    for y in range(0,len(matname)-1,1):
#        scipy.sparse.save_npz(os.path.join(dirname_coo, expname[x], expname[x]+'_mg', matname[y]+'.npz'), adatas_mg[x].layers[matname[y]])

In [ ]:
adatas[3]

In [ ]:
adatas_mg[3]

In [ ]:
adatas[3].obs

In [ ]:
adatas_mg[3].obs

In [ ]:
# カウントデータの.obsに、annotationを結合
assayname = ['A_fw', 'A_rev', 'C_fw', 'C_rev', 'G_fw', 'G_rev', 'T_fw', 'T_rev']
r = pd.merge(adatas[3].obs, adatas_mg[3].obs, how="inner", on = "barcode")

In [ ]:
r

In [ ]:
adatas_mg[3].X

In [ ]:
a = pd.read_csv('mtscATAC-seq/mtscATACpaper_large_data_files/source/cellranger_output/CLL_PT1_CD19neg_v12-mtMask_fragments.tsv', sep='\t')

In [ ]:
a

In [ ]:
b = pd.read_csv('mtscATAC-seq/mtscATACpaper_reproducibility-master/figure_CLL/data/singlecell_sumstats/CLL_PT1_CD19neg_v12-mtMask_singlecell.csv')

In [ ]:
b